# 02 - Modeling
## Baselines
- Goal: set the bar that XGBoost/LightGBM must beat. 
    - Two baselines: 
        - (1) always predict the average strength
        - (2) linear regression.

- Only `dev.csv` is used here. `test.csv` stays untouched until the very end.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold

from src.evaluate import regression_metrics
from src.features import DEV_PATH, FEATURES, TARGET

dev = pd.read_csv(DEV_PATH)
X, y, groups = dev[FEATURES], dev[TARGET], dev["recipe_id"]
print("dev->shape:", dev.shape," | X->shape:", X.shape, " | y->shape:", y.shape, " | groups->shape:", groups.shape)

dev->shape: (799, 10)  | X->shape: (799, 8)  | y->shape: (799,)  | groups->shape: (799,)


In [2]:
# 5-fold CV where each recipe stays inside one fold (same leakage rule as the dev/test split)
gkf = GroupKFold(n_splits=5)
rows = []
for fold, (tr, va) in enumerate(gkf.split(X, y, groups), start=1):
    y_tr, y_va = y.iloc[tr], y.iloc[va]
    preds = {
        "mean_predictor": np.full(len(va), y_tr.mean()),
        "linear_regression": LinearRegression().fit(X.iloc[tr], y_tr).predict(X.iloc[va]),
    }
    for name, p in preds.items():
        rows.append({"model": name, "fold": fold, **regression_metrics(y_va, p)})
cv = pd.DataFrame(rows)
cv.round(2)

,model,fold,rmse,mae,r2
0,mean_predictor,1,15.19,12.10,-0.00
1,linear_regression,1,10.04,7.87,0.56
2,mean_predictor,2,15.87,12.77,-0.00
3,linear_regression,2,10.36,8.30,0.57
4,mean_predictor,3,16.63,13.34,-0.00
5,linear_regression,3,11.45,9.13,0.53
6,mean_predictor,4,14.82,12.13,-0.01
7,linear_regression,4,9.25,7.41,0.61
8,mean_predictor,5,16.44,13.67,-0.00
9,linear_regression,5,10.56,8.44,0.59


In [3]:
# mean and spread across the 5 folds
cv.groupby("model")[["rmse", "mae", "r2"]].agg(["mean", "std"]).round(2)

rmse          mae          r2      
                    mean   std   mean   std  mean   std
model                                                  
linear_regression  10.33  0.80   8.23  0.65  0.57  0.03
mean_predictor     15.79  0.78  12.80  0.70 -0.00  0.00

## A fairer linear baseline
- Linear regression can only draw straight lines, but age gains strength along a curve. Before comparing with XGBoost, give the linear model two domain-driven features so the comparison is fair: `log_age` and `water_cement_ratio`. We add them one at a time to see which one matters.

In [4]:
from src.features import ENGINEERED, add_features

dev_fe = add_features(dev)

def cv_linear(cols):
    rows = []
    for tr, va in gkf.split(dev_fe, y, groups):
        model = LinearRegression().fit(dev_fe.iloc[tr][cols], y.iloc[tr])
        pred = model.predict(dev_fe.iloc[va][cols])
        rows.append(regression_metrics(y.iloc[va], pred))
    return pd.DataFrame(rows).mean()

variants = {
    "raw 8 features": FEATURES,
    "+ log_age": FEATURES + ["log_age"],
    "+ water_cement_ratio": FEATURES + ["water_cement_ratio"],
    "+ both": FEATURES + ENGINEERED,
}
pd.DataFrame({name: cv_linear(cols) for name, cols in variants.items()}).T.round(2)

,rmse,mae,r2
raw 8 features,10.33,8.23,0.57
+ log_age,7.08,5.55,0.80
+ water_cement_ratio,10.30,8.15,0.57
+ both,7.03,5.54,0.80
